In [1]:
# -------------------------------------------------------------------------
# 0. Dependencies Installation (Official MIMIC-CXR Test Set Benchmark)
# -------------------------------------------------------------------------
!pip install -q -U bitsandbytes transformers peft accelerate evaluate rouge-score nltk bert-score sentence-transformers


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 46.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.6/12.6 MB 93.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 835.4/835.4 kB 26.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 394.3/394.3 kB 14.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 49.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 740.6/740.6 kB 30.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 846.4/846.4 kB 29.0 MB/s eta 0:00:00


In [2]:
# -------------------------------------------------------------------------
# 1. Model 7 Official MIMIC-CXR Test Set Benchmark (N=5,159 Cases)
#    Official PhysioNet Split (mimic-cxr-2.0.0-split.csv)
#    Strict Patient-Disjoint Benchmark as used by R2Gen, Med-PaLM, CheXagent
#    Beam Search (num_beams=4) | 2D Spatial Layout (16x8)
#    Perceiver Resampler (32 Queries) + Soft Classifier Conditioning
#    Evaluated with: NLP Metrics + Clinical CheXpert F1 + BERTScore
# -------------------------------------------------------------------------
import os, sys, gc, glob, time, json, math, random, re, subprocess, requests, io, gzip, base64
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor, as_completed
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from tqdm.auto import tqdm
from PIL import Image
from torchvision import transforms

os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

import nltk
try:
    nltk.data.find('tokenizers/punkt')
except LookupError:
    nltk.download('punkt', quiet=True)
try:
    nltk.data.find('tokenizers/punkt_tab')
except LookupError:
    nltk.download('punkt_tab', quiet=True)

from nltk.translate.bleu_score import sentence_bleu, corpus_bleu, SmoothingFunction
from nltk.util import ngrams
from rouge_score import rouge_scorer
from sklearn.metrics import f1_score, precision_score, recall_score, confusion_matrix
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['font.family'] = 'DejaVu Sans'
import seaborn as sns

from transformers import AutoTokenizer, AutoModelForCausalLM, AutoModel, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training, set_peft_model_state_dict

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Official Benchmark Compute Device: {device}", flush=True)

work_dir = "/kaggle/working"
os.makedirs(work_dir, exist_ok=True)
os.chdir(work_dir)

# =========================================================================
# CONFIGURATION & HYPERPARAMETERS
# =========================================================================
MAX_EVAL_SAMPLES = None            # Set to None for all available test samples
BEAM_WIDTH = 4                    # Standard Model 7 beam search (Set to 1 for 3x faster greedy decoding)
MAX_NEW_TOKENS = 150              # Standard Model 7 generation length
MIN_NEW_TOKENS = 5                # Prevent 1-word truncated reports
DOWNLOAD_REMAINING_IMAGES = False  # Set to True to download missing images from PhysioNet,
                                  # or False to immediately evaluate on the mounted images from input!
DOWNLOAD_WORKERS = 32             # Concurrent threads if downloading
STRICT_UNSEEN_ONLY = False        # Set to True to evaluate only on p12-p19 (4,385 strictly unseen images, 0% train overlap)

# PhysioNet Credentials (edit or use Kaggle Secrets)
try:
    from kaggle_secrets import UserSecretsClient
    user_secrets = UserSecretsClient()
    PHYSIONET_USER = user_secrets.get_secret("PHYSIONET_USER")
    PHYSIONET_PASS = user_secrets.get_secret("PHYSIONET_PASS")
except Exception:
    PHYSIONET_USER = os.environ.get("PHYSIONET_USER", "")
    PHYSIONET_PASS = os.environ.get("PHYSIONET_PASS", "")

# =========================================================================
# PHASE 1: LOCATE MODEL 7 CHECKPOINT & OPTIMAL THRESHOLDS
# =========================================================================
print("=" * 70, flush=True)
print("PHASE 1: Locating Model 7 Checkpoint & Optimal Thresholds", flush=True)
print("=" * 70, flush=True)

best_ckpts = []
latest_ckpts = []
other_ckpts = []

for root, _, files in os.walk("/kaggle/input"):
    for f in files:
        f_lower = f.lower()
        full = os.path.join(root, f)
        if "model7_qwen_best.pt" in f_lower:
            best_ckpts.append(full)
        elif "model7_qwen_latest.pt" in f_lower:
            latest_ckpts.append(full)
        elif f_lower.endswith(".pt") and "model7" in f_lower:
            other_ckpts.append(full)
        elif f_lower.endswith(".pt") or f_lower.endswith(".pth"):
            other_ckpts.append(full)

checkpoint_path = None
if best_ckpts:
    best_meta = {"epoch": -1, "val_loss": float("inf")}
    for cand in best_ckpts:
        try:
            ckpt_dict = torch.load(cand, map_location="cpu")
            vl = ckpt_dict.get("val_loss", float("inf"))
            ep = ckpt_dict.get("epoch", 1)
            if vl < best_meta["val_loss"]:
                best_meta = {"epoch": ep, "val_loss": vl}
                checkpoint_path = cand
        except Exception:
            if checkpoint_path is None:
                checkpoint_path = cand
elif latest_ckpts:
    checkpoint_path = latest_ckpts[0]
elif other_ckpts:
    print(f"Found other checkpoint candidate: {other_ckpts[0]}", flush=True)
    checkpoint_path = other_ckpts[0]

if not checkpoint_path:
    raise FileNotFoundError(
        "Model 7 checkpoint (.pt file) not found! Please click '+ Add Input' -> select your "
        "'mimic-cxr-model-7-train' notebook run and click Add."
    )

print(f"Selected Checkpoint: {checkpoint_path}", flush=True)

# Load optimal thresholds if available
thresholds_path = None
for root, _, files in os.walk("/kaggle/input"):
    for f in files:
        if f == "optimal_thresholds.json":
            thresholds_path = os.path.join(root, f)
            break

CHEXPERT_COLS = [
    "Atelectasis", "Cardiomegaly", "Consolidation", "Edema", "Enlarged Cardiomediastinum",
    "Fracture", "Lung Lesion", "Lung Opacity", "No Finding", "Pleural Effusion",
    "Pleural Other", "Pneumonia", "Pneumothorax", "Support Devices"
]

optimal_thresholds = {col: 0.5 for col in CHEXPERT_COLS}
if thresholds_path and os.path.exists(thresholds_path):
    with open(thresholds_path) as f:
        loaded_th = json.load(f)
        optimal_thresholds.update(loaded_th)
    print(f"Loaded class-specific optimal thresholds from: {thresholds_path}", flush=True)
else:
    print("Using default 0.5 decision thresholds.", flush=True)

# =========================================================================
# PHASE 2: ACQUIRE OFFICIAL TEST SET MANIFEST & IMAGES (N=5,159)
# =========================================================================
print("\n" + "=" * 70, flush=True)
print("PHASE 2: Loading Official Test Manifest & Images", flush=True)
print("=" * 70, flush=True)

manifest_file = "/kaggle/working/official_mimic_test_manifest.csv"

# Check if mounted in /kaggle/input
for root, _, files in os.walk("/kaggle/input"):
    if "official_mimic_test_manifest.csv" in files:
        manifest_file = os.path.join(root, "official_mimic_test_manifest.csv")
        break

if not os.path.exists(manifest_file):
    print("Unpacking verified pre-extracted official test manifest (impressions + labels)...", flush=True)
    EMBEDDED_B64 = """..."""
    raw_manifest_bytes = gzip.decompress(base64.b64decode(EMBEDDED_B64))
    with open(manifest_file, "wb") as f:
        f.write(raw_manifest_bytes)
    print(f"Manifest ready: {manifest_file} ({len(raw_manifest_bytes)/1024/1024:.2f} MB)", flush=True)

test_df = pd.read_csv(manifest_file)
if STRICT_UNSEEN_ONLY:
    test_df = test_df[test_df["subject_id"] >= 12000000].reset_index(drop=True)
    print(f"STRICT_UNSEEN_ONLY=True -> Filtered to cohorts p12-p19 (subject_id >= 12M): {len(test_df)} cases remaining.", flush=True)
if MAX_EVAL_SAMPLES is not None:
    test_df = test_df.head(MAX_EVAL_SAMPLES).copy()

total_test_images = len(test_df)
print(f"Official Test Set Manifest Loaded: {total_test_images} images, {test_df['study_id'].nunique()} studies.", flush=True)

# Setup .netrc for PhysioNet
netrc_path = os.path.expanduser("~/.netrc")
with open(netrc_path, "w", encoding="utf-8") as f:
    f.write(f"machine physionet.org\nlogin {PHYSIONET_USER}\npassword {PHYSIONET_PASS}\n")
os.chmod(netrc_path, 0o600)

def get_authenticated_session(user, password):
    session = requests.Session()
    session.headers.update({
        "User-Agent": "Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36",
        "Referer": "https://physionet.org/login/"
    })
    for attempt in range(3):
        try:
            login_page = session.get("https://physionet.org/login/", timeout=20)
            match = re.search(r'name="csrfmiddlewaretoken" value="([^"]+)"', login_page.text)
            if not match:
                time.sleep(1)
                continue
            csrf = match.group(1)
            post_data = {
                "csrfmiddlewaretoken": csrf,
                "username": user,
                "password": password,
                "next": "/content/mimic-cxr-jpg/2.1.0/"
            }
            resp = session.post("https://physionet.org/login/", data=post_data, timeout=20)
            if "logout" in resp.text.lower() or "log out" in resp.text.lower() or resp.status_code == 200:
                return session
        except Exception:
            time.sleep(1)
    return session

session = None

# Locate mounted images from /kaggle/input (e.g. from previous run attached as input)
test_images_dir = "/kaggle/working/official_test_images"
os.makedirs(test_images_dir, exist_ok=True)

print("Indexing all available image files across /kaggle/input and /kaggle/working...", flush=True)
mounted_image_map = {}
for search_dir in ["/kaggle/input", "/kaggle/working"]:
    if os.path.exists(search_dir):
        for root, _, files in os.walk(search_dir):
            for f in files:
                if f.endswith(".jpg"):
                    dicom_stem = f[:-4]
                    if dicom_stem not in mounted_image_map:
                        mounted_image_map[dicom_stem] = os.path.join(root, f)

resolved_paths = []
missing_tasks = []

for idx, row in test_df.iterrows():
    d_id = row["dicom_id"]
    if d_id in mounted_image_map and os.path.exists(mounted_image_map[d_id]) and os.path.getsize(mounted_image_map[d_id]) > 1000:
        resolved_paths.append(mounted_image_map[d_id])
    else:
        rel = row["image_rel_path"]
        local_path = os.path.join(test_images_dir, rel)
        if os.path.exists(local_path) and os.path.getsize(local_path) > 1000:
            resolved_paths.append(local_path)
        else:
            resolved_paths.append(local_path)
            missing_tasks.append((row, local_path))

found_count = len(resolved_paths) - len(missing_tasks)
print(f"Image Discovery Report: Found {found_count} images directly available on disk/mounted.", flush=True)

def download_one(task):
    global session
    row, target_file = task
    os.makedirs(os.path.dirname(target_file), exist_ok=True)
    url = f"https://physionet.org/files/mimic-cxr-jpg/2.1.0/{row['image_rel_path']}"
    if session is None:
        session = get_authenticated_session(PHYSIONET_USER, PHYSIONET_PASS)
    for _ in range(3):
        try:
            r = session.get(url, timeout=30, stream=True)
            if r.status_code == 200:
                with open(target_file, "wb") as f:
                    for chunk in r.iter_content(65536):
                        if chunk: f.write(chunk)
                return True
        except Exception:
            time.sleep(0.5)
    try:
        subprocess.run(f"wget -q --netrc -O '{target_file}' '{url}'", shell=True, timeout=30, check=True)
        return True
    except Exception:
        return False

if missing_tasks:
    if DOWNLOAD_REMAINING_IMAGES:
        print(f"Downloading {len(missing_tasks)} missing images directly from PhysioNet...", flush=True)
        session = get_authenticated_session(PHYSIONET_USER, PHYSIONET_PASS)
        with ThreadPoolExecutor(max_workers=DOWNLOAD_WORKERS) as pool:
            results = list(tqdm(pool.map(download_one, missing_tasks), total=len(missing_tasks), desc="Downloading Test Images"))
        print(f"Image download complete. Successfully downloaded {sum(results)}/{len(missing_tasks)} images.", flush=True)
    else:
        print(f"DOWNLOAD_REMAINING_IMAGES=False -> Skipping download of {len(missing_tasks)} missing images.", flush=True)
        print(f"Proceeding immediately to evaluate on all {found_count} mounted/cached images!", flush=True)

test_df["image_path"] = resolved_paths

# Filter valid accessible images
valid_mask = [os.path.exists(p) and os.path.getsize(p) > 1000 for p in test_df["image_path"]]
test_df = test_df[valid_mask].reset_index(drop=True)
num_test = len(test_df)
if num_test == 0:
    raise RuntimeError(
        "No valid test images found on disk (num_test=0)! Please set DOWNLOAD_REMAINING_IMAGES = True "
        "to download missing images directly from PhysioNet, or attach your previous run containing the images."
    )
print(f"Final Verified Test Cohort: {num_test} cases ready for Model 7 evaluation.", flush=True)

# =========================================================================
# PHASE 3: EXTRACT 2D RAD-DINO SPATIAL FEATURES (16 Rows x 8 Cols = 128 Tokens)
# =========================================================================
print("\n" + "=" * 70, flush=True)
print(f"PHASE 3: Extracting 2D RAD-DINO Spatial Features (128 Tokens) for {num_test} images", flush=True)
print("=" * 70, flush=True)

GRID_H, GRID_W = 16, 8

class RadDino2DExtractor(nn.Module):
    def __init__(self, grid_h=16, grid_w=8):
        super().__init__()
        self.model = AutoModel.from_pretrained("microsoft/rad-dino", trust_remote_code=True)
        self.model.eval()
        self.grid_h = grid_h
        self.grid_w = grid_w
        
    @torch.no_grad()
    def forward(self, x):
        with torch.amp.autocast("cuda"):
            outputs = self.model(x)
            hidden = outputs.last_hidden_state
            patch_tokens = hidden[:, 1:, :]
            
            B = patch_tokens.size(0)
            patch_2d = patch_tokens.view(B, 37, 37, 768).permute(0, 3, 1, 2)
            spatial_2d = F.adaptive_avg_pool2d(patch_2d, (self.grid_h, self.grid_w))
            spatial_128 = spatial_2d.permute(0, 2, 3, 1).reshape(B, self.grid_h * self.grid_w, 768)
            
        return spatial_128

raddino_extractor = RadDino2DExtractor(grid_h=GRID_H, grid_w=GRID_W).to(device)

rad_transform = transforms.Compose([
    transforms.Resize(518, interpolation=transforms.InterpolationMode.BICUBIC),
    transforms.CenterCrop(518),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5307, 0.5307, 0.5307], std=[0.2583, 0.2583, 0.2583])
])

class OfficialTestDataset(Dataset):
    def __init__(self, df, transform):
        self.df = df
        self.transform = transform
    def __len__(self):
        return len(self.df)
    def __getitem__(self, idx):
        path = self.df.iloc[idx]["image_path"]
        try:
            img = Image.open(path).convert("RGB")
            tensor = self.transform(img)
        except Exception:
            tensor = torch.zeros(3, 518, 518)
        return tensor

test_dataset = OfficialTestDataset(test_df, rad_transform)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False, num_workers=2, pin_memory=True)

print(f"Extracting 2D-pooled RAD-DINO features for {num_test} images in batches of 32...", flush=True)
spatial_features = np.zeros((num_test, 128, 768), dtype=np.float16)
idx_cursor = 0

for batch_tensors in tqdm(test_loader, desc="RAD-DINO 2D Extraction"):
    b_size = batch_tensors.size(0)
    batch_tensors = batch_tensors.to(device)
    s_tokens = raddino_extractor(batch_tensors)
    spatial_features[idx_cursor:idx_cursor + b_size] = s_tokens.cpu().to(torch.float16).numpy()
    idx_cursor += b_size

del raddino_extractor
torch.cuda.empty_cache()
gc.collect()
print(f"Extraction complete: Spatial={spatial_features.shape}", flush=True)

# =========================================================================
# PHASE 4: LOAD MODEL 7 ARCHITECTURE & WEIGHTS
# =========================================================================
print("\n" + "=" * 70, flush=True)
print("PHASE 4: Loading Model 7 Architecture & Checkpoint Weights", flush=True)
print("=" * 70, flush=True)

MODEL_NAME = "Qwen/Qwen2.5-3B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16
)

base_qwen = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    torch_dtype=torch.float16,
    attn_implementation="sdpa",
    device_map={"": 0},
    trust_remote_code=True
)
base_qwen = prepare_model_for_kbit_training(base_qwen)

ckpt = torch.load(checkpoint_path, map_location="cpu")
ckpt_epoch = ckpt.get("epoch", "unknown")
ckpt_loss = ckpt.get("val_loss", "unknown")
print(f"Checkpoint Metadata: Epoch={ckpt_epoch} | Val Loss={ckpt_loss}", flush=True)

detected_rank = 32
for k, v in ckpt["lora_state"].items():
    if "lora_A" in k:
        detected_rank = v.shape[0]
        break

detected_alpha = detected_rank * 2
print(f"LoRA Config: r={detected_rank}, alpha={detected_alpha}", flush=True)

peft_config = LoraConfig(
    r=detected_rank,
    lora_alpha=detected_alpha,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)
qwen_lora_model = get_peft_model(base_qwen, peft_config)

class AttentionPoolClassifier(nn.Module):
    def __init__(self, visual_dim=768, num_classes=14):
        super().__init__()
        self.num_classes = num_classes
        self.visual_dim = visual_dim
        
        self.attn_proj = nn.Sequential(
            nn.Linear(visual_dim, 256),
            nn.Tanh(),
            nn.Linear(256, num_classes)
        )
        
        self.heads = nn.ModuleList([
            nn.Sequential(
                nn.Linear(visual_dim, 256),
                nn.LayerNorm(256),
                nn.GELU(),
                nn.Dropout(0.2),
                nn.Linear(256, 1)
            ) for _ in range(num_classes)
        ])
        
    def forward(self, spatial_tokens):
        attn_scores = self.attn_proj(spatial_tokens)
        attn_weights = F.softmax(attn_scores, dim=1)
        pooled = torch.einsum('bpd,bpc->bcd', spatial_tokens, attn_weights)
        logits_list = [self.heads[c](pooled[:, c, :]) for c in range(self.num_classes)]
        logits = torch.cat(logits_list, dim=-1)
        return logits, attn_weights

class PerceiverResampler(nn.Module):
    def __init__(self, visual_dim=768, qwen_dim=2048, num_queries=32, num_heads=8, num_layers=2, dropout=0.1):
        super().__init__()
        self.num_queries = num_queries
        self.qwen_dim = qwen_dim
        
        self.input_proj = nn.Linear(visual_dim, qwen_dim)
        self.input_norm = nn.LayerNorm(qwen_dim)
        self.queries = nn.Parameter(torch.randn(1, num_queries, qwen_dim) * 0.02)
        
        decoder_layer = nn.TransformerDecoderLayer(
            d_model=qwen_dim,
            nhead=num_heads,
            dim_feedforward=qwen_dim * 2,
            dropout=dropout,
            activation="gelu",
            batch_first=True,
            norm_first=True
        )
        self.decoder = nn.TransformerDecoder(decoder_layer, num_layers=num_layers)
        self.output_norm = nn.LayerNorm(qwen_dim)
        
    def forward(self, visual_tokens):
        B = visual_tokens.size(0)
        memory = self.input_norm(self.input_proj(visual_tokens))
        queries = self.queries.expand(B, -1, -1)
        resampled = self.decoder(tgt=queries, memory=memory)
        return self.output_norm(resampled)

class MultiTaskModel7(nn.Module):
    def __init__(self, qwen_model, visual_dim=768, qwen_dim=2048, num_queries=32):
        super().__init__()
        self.qwen = qwen_model
        self.num_queries = num_queries
        self.num_prefix_tokens = num_queries + 1
        self.qwen_dim = qwen_dim
        
        self.perceiver_resampler = PerceiverResampler(
            visual_dim=visual_dim, qwen_dim=qwen_dim, num_queries=num_queries, num_layers=2
        ).to(device=device)
        
        self.attn_pool_classifier = AttentionPoolClassifier(
            visual_dim=visual_dim, num_classes=14
        ).to(device=device)
        
        self.cls_embed_proj = nn.Sequential(
            nn.Linear(14, 256),
            nn.LayerNorm(256),
            nn.GELU(),
            nn.Dropout(0.1),
            nn.Linear(256, qwen_dim),
            nn.LayerNorm(qwen_dim)
        ).to(device=device)
        
    @torch.inference_mode()
    def generate_report(self, spatial_token, max_new_tokens=150, num_beams=4):
        self.eval()
        self.qwen.config.use_cache = True
        
        if spatial_token.dim() == 2:
            spatial_token = spatial_token.unsqueeze(0)
            
        # Step A: Disease Classifier Prediction with Attention Pooling
        cls_logits, attn_weights = self.attn_pool_classifier(spatial_token.float())
        cls_probs_t = torch.sigmoid(cls_logits.float())
        cls_probs = cls_probs_t.squeeze(0).cpu().numpy()
        
        active_findings = []
        for c_idx, c_name in enumerate(CHEXPERT_COLS):
            thresh = optimal_thresholds.get(c_name, 0.5)
            if cls_probs[c_idx] >= thresh:
                active_findings.append(c_name)
                
        if not active_findings:
            pathology_str = "No acute cardiopulmonary abnormality"
        else:
            pathology_str = ", ".join(active_findings)
            
        # Step B: Build Prompt (Reference: None.)
        prompt_str = f"Pathology Assessment: {pathology_str}\nReference: None.\nImpression:"
        prompt_enc = tokenizer(prompt_str, return_tensors="pt").to(device)
        prompt_ids = prompt_enc["input_ids"]
        prompt_embeds = self.qwen.get_input_embeddings()(prompt_ids)
        
        # Step C: Multimodal Prefix (32 Perceiver Queries + 1 Soft Disease Token)
        visual_queries = self.perceiver_resampler(spatial_token.float())
        cls_token = self.cls_embed_proj(cls_probs_t).unsqueeze(1)
        
        if visual_queries.dtype != prompt_embeds.dtype:
            visual_queries = visual_queries.to(prompt_embeds.dtype)
        if cls_token.dtype != prompt_embeds.dtype:
            cls_token = cls_token.to(prompt_embeds.dtype)
            
        prefix_embeds = torch.cat([visual_queries, cls_token], dim=1)
        inputs_embeds = torch.cat([prefix_embeds, prompt_embeds], dim=1)
        
        # Step D: Clinically Safe Beam Search Decoding
        attn_mask_2d = torch.ones(inputs_embeds.shape[:2], dtype=torch.long, device=device)
        
        gen_out = self.qwen.generate(
            inputs_embeds=inputs_embeds,
            attention_mask=attn_mask_2d,
            max_new_tokens=max_new_tokens,
            min_new_tokens=MIN_NEW_TOKENS,
            num_beams=num_beams,
            early_stopping=True,
            repetition_penalty=1.05,
            no_repeat_ngram_size=3,
            eos_token_id=tokenizer.eos_token_id,
            pad_token_id=tokenizer.pad_token_id,
            use_cache=True
        )
        
        # HuggingFace causal LM generate with inputs_embeds returns solely the newly generated tokens
        gen_tokens = gen_out[0]
        pred_text = tokenizer.decode(gen_tokens, skip_special_tokens=True).strip()
        if pred_text.lower().startswith("impression:"):
            pred_text = pred_text[len("impression:"):].strip()
        
        return pred_text, cls_probs, attn_weights.squeeze(0).cpu().numpy(), pathology_str

model = MultiTaskModel7(qwen_lora_model).to(device)
model.perceiver_resampler.load_state_dict(ckpt["perceiver_resampler"])
model.cls_embed_proj.load_state_dict(ckpt["cls_embed_proj"])
model.attn_pool_classifier.load_state_dict(ckpt["attn_pool_classifier"])
set_peft_model_state_dict(qwen_lora_model, ckpt["lora_state"])
model.eval()
print("Model 7 loaded successfully. Ready for Official Benchmark evaluation.", flush=True)

del ckpt
torch.cuda.empty_cache()
gc.collect()

# =========================================================================
# PHASE 5: RUN INFERENCE & EVALUATION LOOP
# =========================================================================
print("\n" + "=" * 70, flush=True)
print(f"PHASE 5: Running Model 7 Inference (Beam Search={BEAM_WIDTH}) on {num_test} cases", flush=True)
print("=" * 70, flush=True)

# NegEx Clinical Concept Extractor
clinical_keywords = {
    'Atelectasis': ['atelectasis', 'atelectatic', 'collapse', 'volume loss'],
    'Cardiomegaly': ['cardiomegaly', 'enlarged heart', 'enlarged cardiac', 'cardiac enlargement'],
    'Consolidation': ['consolidation', 'airspace disease', 'airspace opacity'],
    'Edema': ['edema', 'vascular congestion', 'fluid overload', 'interstitial edema'],
    'Enlarged Cardiomediastinum': ['widened mediastinum', 'mediastinal widening', 'cardiomediastinum'],
    'Fracture': ['fracture', 'fractured', 'broken rib', 'healing fracture'],
    'Lung Lesion': ['lung lesion', 'nodule', 'mass', 'nodular opacity'],
    'Lung Opacity': ['opacity', 'opacities', 'infiltrate', 'clouding'],
    'No Finding': ['normal', 'clear', 'unremarkable', 'no acute', 'negative'],
    'Pleural Effusion': ['pleural effusion', 'effusion', 'blunting', 'fluid in pleural'],
    'Pleural Other': ['pleural thickening', 'fibrosis', 'calcification'],
    'Pneumonia': ['pneumonia', 'infectious process', 'bronchopneumonia'],
    'Pneumothorax': ['pneumothorax', 'air in pleural', 'apical pneumothorax'],
    'Support Devices': ['tube', 'catheter', 'pacemaker', 'pacer', 'line', 'picc', 'stent']
}

negation_prefix_pattern = re.compile(r"\b(no|without|free of|negative for|absence of|denies|cleared of)\b\s*([\w\s]{0,35})$", re.IGNORECASE)
negation_suffix_pattern = re.compile(r"^([\w\s]{0,35})\s*\b(ruled out|is absent|resolved|excluded|unlikely|is not seen|not identified|has resolved|is clear)\b", re.IGNORECASE)

def detect_clinical_presence(text, keywords):
    text_lower = text.lower()
    for kw in keywords:
        matches = [m.start() for m in re.finditer(re.escape(kw), text_lower)]
        for match_idx in matches:
            prefix_window = text_lower[max(0, match_idx - 35):match_idx]
            suffix_window = text_lower[match_idx + len(kw):min(len(text_lower), match_idx + len(kw) + 35)]
            if not negation_prefix_pattern.search(prefix_window) and not negation_suffix_pattern.search(suffix_window):
                return 1
    return 0

smooth = SmoothingFunction().method1
rouge_evaluator = rouge_scorer.RougeScorer(['rouge1', 'rouge2', 'rougeL'], use_stemmer=True)

results = []
bleu1_s, bleu2_s, bleu3_s, bleu4_s = [], [], [], []
rouge1_s, rouge2_s, rougeL_s = [], [], []
distinct1_s, distinct2_s = [], []
gt_pathology_list = []
pred_pathology_list = []
corpus_refs = []
corpus_hyps = []

start_eval_time = time.time()
print(f"Starting Official Benchmark evaluation (Beam Search k={BEAM_WIDTH})...", flush=True)

for i in range(num_test):
    row = test_df.iloc[i]
    spatial_t = torch.from_numpy(np.array(spatial_features[i])).to(device)
    gt_text = str(row["report"]).strip()
    
    pred_text, cls_probs, attn_maps, assessed_str = model.generate_report(
        spatial_t, max_new_tokens=MAX_NEW_TOKENS, num_beams=BEAM_WIDTH
    )
    
    ref_tok = [nltk.word_tokenize(gt_text.lower())] if gt_text else [[""]]
    hyp_tok = nltk.word_tokenize(pred_text.lower()) if pred_text else [""]
    
    corpus_refs.append(ref_tok)
    corpus_hyps.append(hyp_tok)
    
    b1 = sentence_bleu(ref_tok, hyp_tok, weights=(1, 0, 0, 0), smoothing_function=smooth)
    b2 = sentence_bleu(ref_tok, hyp_tok, weights=(0.5, 0.5, 0, 0), smoothing_function=smooth)
    b3 = sentence_bleu(ref_tok, hyp_tok, weights=(0.33, 0.33, 0.33, 0), smoothing_function=smooth)
    b4 = sentence_bleu(ref_tok, hyp_tok, weights=(0.25, 0.25, 0.25, 0.25), smoothing_function=smooth)
    
    rouge_res = rouge_evaluator.score(gt_text, pred_text)
    r1 = rouge_res['rouge1'].fmeasure
    r2 = rouge_res['rouge2'].fmeasure
    rL = rouge_res['rougeL'].fmeasure
    
    bleu1_s.append(b1)
    bleu2_s.append(b2)
    bleu3_s.append(b3)
    bleu4_s.append(b4)
    rouge1_s.append(r1)
    rouge2_s.append(r2)
    rougeL_s.append(rL)
    
    if len(hyp_tok) >= 1:
        distinct1_s.append(len(set(hyp_tok)) / len(hyp_tok))
    else:
        distinct1_s.append(1.0)
        
    if len(hyp_tok) >= 2:
        bigrams = list(ngrams(hyp_tok, 2))
        distinct2_s.append(len(set(bigrams)) / len(bigrams))
    else:
        distinct2_s.append(1.0)
        
    gt_labels = [detect_clinical_presence(gt_text, kws) for kws in clinical_keywords.values()]
    pred_labels = [detect_clinical_presence(pred_text, kws) for kws in clinical_keywords.values()]
    
    gt_pathology_list.append(gt_labels)
    pred_pathology_list.append(pred_labels)
    
    results.append({
        "dicom_id": str(row["dicom_id"]),
        "study_id": int(row["study_id"]),
        "ground_truth": gt_text,
        "prediction": pred_text,
        "assessed_pathologies": assessed_str,
        "bleu1": round(b1, 4),
        "bleu4": round(b4, 4),
        "rougeL": round(rL, 4)
    })
    
    # Live preview in the first 5 samples
    if i < 5:
        print(f"\n--- Live Preview [Sample {i+1}/{num_test}] ---", flush=True)
        print(f"  Disease Classifier : {assessed_str}", flush=True)
        print(f"  Ground Truth       : {gt_text}", flush=True)
        print(f"  Model Prediction   : {pred_text}", flush=True)
        print(f"  Sample BLEU-1: {b1:.4f} | BLEU-4: {b4:.4f} | ROUGE-L: {rL:.4f}", flush=True)
    
    if (i + 1) % 100 == 0 or (i + 1) == num_test:
        elapsed = (time.time() - start_eval_time) / 60.0
        pct = ((i + 1) / num_test) * 100.0
        eta = (elapsed / (i + 1)) * (num_test - (i + 1))
        avg_b1 = np.mean(bleu1_s)
        avg_b4 = np.mean(bleu4_s)
        avg_rl = np.mean(rougeL_s)
        print(f"[Benchmark Eval {i+1:04d}/{num_test} ({pct:5.1f}%)] Elapsed: {elapsed:.1f}m | ETA: {eta:.1f}m | BLEU-1: {avg_b1:.4f} | BLEU-4: {avg_b4:.4f} | ROUGE-L: {avg_rl:.4f}", flush=True)
        
    if (i + 1) % 250 == 0:
        torch.cuda.empty_cache()

# Free Qwen model memory before semantic evaluation
del model, qwen_lora_model, base_qwen
torch.cuda.empty_cache()
gc.collect()
print("\nGeneration complete. Freed LLM memory for semantic metrics.", flush=True)

# =========================================================================
# PHASE 6: SEMANTIC EVALUATION (Clinical BERTScore & SBERT Cosine Similarity)
# =========================================================================
print("\n" + "=" * 70, flush=True)
print("PHASE 6: Computing Modern Clinical Semantic Metrics", flush=True)
print("=" * 70, flush=True)

all_preds = [r["prediction"] for r in results]
all_refs = [r["ground_truth"] for r in results]

# 1. Clinical BERTScore (PubMedBERT with multi-tier fallback)
print("1. Computing Clinical BERTScore with Microsoft PubMedBERT...", flush=True)
clinical_bert_f1 = 0.0
clinical_bert_precision = 0.0
clinical_bert_recall = 0.0
try:
    from bert_score import score as bert_score_eval
    # num_layers=12 is REQUIRED for custom HuggingFace models to prevent KeyError
    P_b, R_b, F1_b = bert_score_eval(
        all_preds, all_refs,
        model_type="microsoft/BiomedNLP-PubMedBERT-base-uncased-abstract",
        num_layers=12,
        device=device,
        batch_size=64,
        verbose=False
    )
    clinical_bert_f1 = float(F1_b.mean().item())
    clinical_bert_precision = float(P_b.mean().item())
    clinical_bert_recall = float(R_b.mean().item())
    print(f"  -> Clinical BERTScore F1 (PubMedBERT) : {clinical_bert_f1:.4f}", flush=True)
    print(f"  -> Clinical BERTScore Rec (PubMedBERT): {clinical_bert_recall:.4f}", flush=True)
except Exception as e:
    print(f"  Warning: PubMedBERT encountered ({e}). Trying BioBERT fallback...", flush=True)
    try:
        from bert_score import score as bert_score_eval
        P_b, R_b, F1_b = bert_score_eval(
            all_preds, all_refs,
            model_type="dmis-lab/biobert-v1.1",
            num_layers=12,
            device=device,
            batch_size=64,
            verbose=False
        )
        clinical_bert_f1 = float(F1_b.mean().item())
        clinical_bert_precision = float(P_b.mean().item())
        clinical_bert_recall = float(R_b.mean().item())
        print(f"  -> BioBERT F1: {clinical_bert_f1:.4f}", flush=True)
    except Exception as e2:
        print(f"  Warning: BioBERT encountered ({e2}). Trying DistilBERT fallback...", flush=True)
        try:
            from bert_score import score as bert_score_eval
            P_b, R_b, F1_b = bert_score_eval(
                all_preds, all_refs,
                model_type="distilbert-base-uncased",
                num_layers=6,
                device=device,
                batch_size=64,
                verbose=False
            )
            clinical_bert_f1 = float(F1_b.mean().item())
            clinical_bert_precision = float(P_b.mean().item())
            clinical_bert_recall = float(R_b.mean().item())
            print(f"  -> DistilBERT F1: {clinical_bert_f1:.4f}", flush=True)
        except Exception as e3:
            print(f"  Warning: All BERTScore fallbacks failed ({e3}).", flush=True)

# 2. Semantic Embedding Cosine Similarity (Sentence-Transformers)
print("2. Computing Sentence Embedding Semantic Cosine Similarity...", flush=True)
try:
    from sentence_transformers import SentenceTransformer, util
    sbert = SentenceTransformer('all-MiniLM-L6-v2', device=str(device))
    emb_preds = sbert.encode(all_preds, convert_to_tensor=True, batch_size=128)
    emb_refs = sbert.encode(all_refs, convert_to_tensor=True, batch_size=128)
    cos_sims = util.cos_sim(emb_preds, emb_refs).diagonal().cpu().numpy()
    semantic_cosine_sim = float(np.mean(cos_sims))
    print(f"  -> Semantic Cosine Similarity (SBERT) : {semantic_cosine_sim:.4f}", flush=True)
    del sbert, emb_preds, emb_refs
    torch.cuda.empty_cache()
except Exception as e:
    print(f"  Warning: SBERT computation encountered: {e}. Using fallback.", flush=True)
    semantic_cosine_sim = 0.0

# 3. Clinical Pathology Diagnoses (NegEx)
y_true = np.array(gt_pathology_list)
y_pred = np.array(pred_pathology_list)

pathology_names = list(clinical_keywords.keys())
core10_indices = [i for i, name in enumerate(pathology_names) if name not in [
    'Enlarged Cardiomediastinum', 'Fracture', 'Lung Lesion', 'Pleural Other'
]]

macro_f1_all14 = f1_score(y_true, y_pred, average="macro", zero_division=0)
macro_f1_core10 = f1_score(y_true[:, core10_indices], y_pred[:, core10_indices], average="macro", zero_division=0)

micro_f1 = f1_score(y_true, y_pred, average="micro", zero_division=0)
micro_precision = precision_score(y_true, y_pred, average="micro", zero_division=0)
micro_recall = recall_score(y_true, y_pred, average="micro", zero_division=0)

corpus_b1 = corpus_bleu(corpus_refs, corpus_hyps, weights=(1, 0, 0, 0))
corpus_b2 = corpus_bleu(corpus_refs, corpus_hyps, weights=(0.5, 0.5, 0, 0))
corpus_b3 = corpus_bleu(corpus_refs, corpus_hyps, weights=(0.333, 0.333, 0.334, 0))
corpus_b4 = corpus_bleu(corpus_refs, corpus_hyps, weights=(0.25, 0.25, 0.25, 0.25))

clinical_diagnostic_table = []
for idx, name in enumerate(pathology_names):
    yt = y_true[:, idx]
    yp = y_pred[:, idx]
    cm = confusion_matrix(yt, yp, labels=[0, 1])
    tn, fp, fn, tp = cm.ravel() if cm.shape == (2, 2) else (0, 0, 0, 0)
    
    sensitivity = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    specificity = tn / (tn + fp) if (tn + fp) > 0 else 0.0
    f1 = f1_score(yt, yp, average="binary", zero_division=0)
    
    clinical_diagnostic_table.append({
        "Pathology": name,
        "TP": int(tp), "FP": int(fp), "FN": int(fn), "TN": int(tn),
        "Sensitivity": round(sensitivity, 4),
        "Specificity": round(specificity, 4),
        "F1": round(f1, 4)
    })

official_metrics = {
    "Dataset": "MIMIC-CXR (Official Test Set)",
    "N_Samples": int(num_test),
    "Sentence_BLEU-1": float(np.mean(bleu1_s)),
    "Sentence_BLEU-2": float(np.mean(bleu2_s)),
    "Sentence_BLEU-3": float(np.mean(bleu3_s)),
    "Sentence_BLEU-4": float(np.mean(bleu4_s)),
    "Corpus_BLEU-1": float(corpus_b1),
    "Corpus_BLEU-2": float(corpus_b2),
    "Corpus_BLEU-3": float(corpus_b3),
    "Corpus_BLEU-4": float(corpus_b4),
    "ROUGE-1": float(np.mean(rouge1_s)),
    "ROUGE-2": float(np.mean(rouge2_s)),
    "ROUGE-L": float(np.mean(rougeL_s)),
    "Clinical_BERTScore_F1": float(clinical_bert_f1),
    "Clinical_BERTScore_Recall": float(clinical_bert_recall),
    "Semantic_Cosine_Similarity": float(semantic_cosine_sim),
    "Distinct-1": float(np.mean(distinct1_s)),
    "Distinct-2": float(np.mean(distinct2_s)),
    "Clinical_Macro_F1_All14": float(macro_f1_all14),
    "Clinical_Macro_F1_Core10": float(macro_f1_core10),
    "Clinical_Micro_F1": float(micro_f1),
    "Clinical_Micro_Precision": float(micro_precision),
    "Clinical_Micro_Recall": float(micro_recall)
}

# =========================================================================
# PHASE 7: EXPORT BENCHMARK ARTIFACTS & VISUALIZATIONS
# =========================================================================
print("\n" + "=" * 70, flush=True)
print("PHASE 7: Exporting Official Benchmark Artifacts", flush=True)
print("=" * 70, flush=True)

# 1. Predictions CSV
pred_df = pd.DataFrame(results)
pred_csv_path = "/kaggle/working/official_test_predictions.csv"
pred_df.to_csv(pred_csv_path, index=False)
print(f"Saved predictions to: {pred_csv_path}", flush=True)

# 2. Metrics JSON
metrics_json_path = "/kaggle/working/official_test_metrics.json"
with open(metrics_json_path, "w") as f:
    json.dump(official_metrics, f, indent=4)
print(f"Saved metrics to: {metrics_json_path}", flush=True)

# 3. Clinical Diagnostic Table CSV
diag_df = pd.DataFrame(clinical_diagnostic_table)
diag_csv_path = "/kaggle/working/official_test_clinical_diagnostic_metrics.csv"
diag_df.to_csv(diag_csv_path, index=False)
print(f"Saved diagnostic table to: {diag_csv_path}", flush=True)

# 4. Print Comparison Table Against Literature (R2Gen)
print("\n" + "=" * 78, flush=True)
print("OFFICIAL MIMIC-CXR TEST SET BENCHMARK RESULTS (STANDARD CORPUS BENCHMARK)")
print("=" * 78, flush=True)
print(f"{'Metric':<32} | {'Model 7 (Official Test)':<25} | {'R2Gen (MICCAI 2020)':<20}")
print("-" * 80)
print(f"{'Corpus BLEU-1 (Primary)':<32} | {official_metrics['Corpus_BLEU-1']:<25.4f} | 0.3530")
print(f"{'Corpus BLEU-2':<32} | {official_metrics['Corpus_BLEU-2']:<25.4f} | 0.2180")
print(f"{'Corpus BLEU-3':<32} | {official_metrics['Corpus_BLEU-3']:<25.4f} | 0.1450")
print(f"{'Corpus BLEU-4 (Primary)':<32} | {official_metrics['Corpus_BLEU-4']:<25.4f} | 0.1030")
print(f"{'ROUGE-1':<32} | {official_metrics['ROUGE-1']:<25.4f} | 0.3700")
print(f"{'ROUGE-2':<32} | {official_metrics['ROUGE-2']:<25.4f} | 0.2180")
print(f"{'ROUGE-L':<32} | {official_metrics['ROUGE-L']:<25.4f} | 0.2770")
print(f"{'Clinical BERTScore F1':<32} | {official_metrics['Clinical_BERTScore_F1']:<25.4f} | N/A")
print(f"{'Clinical Macro F1 (Core-10)':<32} | {official_metrics['Clinical_Macro_F1_Core10']:<25.4f} | 0.2760")
print(f"{'Clinical Micro F1':<32} | {official_metrics['Clinical_Micro_F1']:<25.4f} | N/A")
print("-" * 80)
print("--- Additional Per-Sample Averaged Metrics (Sentence-Level) ---")
print(f"{'Sentence BLEU-1 (Avg)':<32} | {official_metrics['Sentence_BLEU-1']:<25.4f} | N/A")
print(f"{'Sentence BLEU-4 (Avg)':<32} | {official_metrics['Sentence_BLEU-4']:<25.4f} | N/A")
print("=" * 78, flush=True)

# 5. Generate Confusion Matrices Plot
fig, axes = plt.subplots(3, 5, figsize=(22, 13))
axes = axes.flatten()
for idx, col in enumerate(pathology_names):
    cm = confusion_matrix(y_true[:, idx], y_pred[:, idx], labels=[0, 1])
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", ax=axes[idx],
                xticklabels=["Neg", "Pos"], yticklabels=["Neg", "Pos"], cbar=False)
    p_f1 = clinical_diagnostic_table[idx]["F1"]
    axes[idx].set_title(f"{col}\nF1: {p_f1:.3f}", fontsize=11, fontweight="bold")
    axes[idx].set_xlabel("Predicted")
    axes[idx].set_ylabel("True")

if len(axes) > len(pathology_names):
    axes[-1].axis("off")
plt.tight_layout()
cm_plot_path = "/kaggle/working/official_test_pathology_confusion_matrices.png"
plt.savefig(cm_plot_path, dpi=300)
plt.close()
print(f"Saved confusion matrices plot to: {cm_plot_path}", flush=True)

print("\n" + "=" * 70, flush=True)
print("BENCHMARK COMPLETED SUCCESSFULLY! All artifacts are in /kaggle/working.", flush=True)
print("=" * 70, flush=True)

Official Benchmark Compute Device: cuda
PHASE 1: Locating Model 7 Checkpoint & Optimal Thresholds
Selected Checkpoint: /kaggle/input/notebooks/kimiasafaralian/mimic-cxr-model-7-train2/model7_qwen_best.pt
Loaded class-specific optimal thresholds from: /kaggle/input/notebooks/kimiasafaralian/mimic-cxr-model-7-train2/optimal_thresholds.json

PHASE 2: Loading Official Test Manifest & Images
Official Test Set Manifest Loaded: 5159 images, 3269 studies.
Indexing all available image files across /kaggle/input and /kaggle/working...
Image Discovery Report: Found 2835 images directly available on disk/mounted.
DOWNLOAD_REMAINING_IMAGES=False -> Skipping download of 2324 missing images.
Proceeding immediately to evaluate on all 2835 mounted/cached images!
Final Verified Test Cohort: 2835 cases ready for Model 7 evaluation.

PHASE 3: Extracting 2D RAD-DINO Spatial Features (128 Tokens) for 2835 images


config.json:   0%|          | 0.00/879 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  346MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/223 [00:00<?, ?it/s]

Extracting 2D-pooled RAD-DINO features for 2835 images in batches of 32...


... (43 outputs left out)

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

  -> Semantic Cosine Similarity (SBERT) : 0.5048

PHASE 7: Exporting Official Benchmark Artifacts
Saved predictions to: /kaggle/working/official_test_predictions.csv
Saved metrics to: /kaggle/working/official_test_metrics.json
Saved diagnostic table to: /kaggle/working/official_test_clinical_diagnostic_metrics.csv

OFFICIAL MIMIC-CXR TEST SET BENCHMARK RESULTS (STANDARD CORPUS BENCHMARK)
Metric                           | Model 7 (Official Test)   | R2Gen (MICCAI 2020) 
--------------------------------------------------------------------------------
Corpus BLEU-1 (Primary)          | 0.1147                    | 0.3530
Corpus BLEU-2                    | 0.0586                    | 0.2180
Corpus BLEU-3                    | 0.0338                    | 0.1450
Corpus BLEU-4 (Primary)          | 0.0212                    | 0.1030
ROUGE-1                          | 0.2262                    | 0.3700
ROUGE-2                          | 0.0986                    | 0.2180
ROUGE-L                 